In [1]:
!pip install langchain_groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 591.4/591.4 kB 8.3 MB/s  0:00:00
  Attempting uninstall: langchain-protocol
    Found existing installation: langchain-protocol 0.0.20
    Uninstalling langchain-protocol-0.0.20:
      Successfully uninstalled langchain-protocol-0.0.20
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.7.4
    Uninstalling langchain-core-1.7.4:
      Successfully uninstalled langchain-core-1.7.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [langchain_groq]


In [2]:
import json
import functools
from openai import RateLimitError, AuthenticationError, APIConnectionError
# Use a model ID currently supported by your Groq account.
MODEL = "openai/gpt-oss-120b"
print("Model configured:", MODEL)

Model configured: openai/gpt-oss-120b


In [5]:
MENU = {
 "plum cake": 500,
 "chocolate cake": 650,
 "cupcake": 80
}
SYSTEM = f'''
ROLE:
You are the friendly assistant for Sweet Crumbs Home Bakery.
FACTS:
Menu and prices in rupees: {json.dumps(MENU)}
Orders need 24 hours notice.
All cakes have eggless versions.
RULES:
Never promise a delivery time.
Never invent information.
If you do not know something, say:
I will check with the owner.
FORMAT:
Reply in simple, friendly language.
Keep answers under 50 words.
'''

In [6]:
import functools

def log_call(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        print(f"[LOG] {func.__name__}{args} -> {result!r}")
        return result
    return wrapper


@log_call
def get_price(item: str, qty: int = 1) -> int:
    """Return the total price in rupees for an item."""
    price = MENU.get(item.lower().strip())

    if price is None:
        return 0

    return price * qty


print("Total for 6 cupcakes:", get_price("cupcake", 6))

[LOG] get_price('cupcake', 6) -> 480
Total for 6 cupcakes: 480


In [8]:
history = [
 {"role": "system", "content": SYSTEM}
]
MAX_TURNS = 8

In [11]:
def chat(user_text: str) -> str:
    history.append({
        "role": "user",
        "content": user_text
    })

    # Keep the system prompt plus the most recent messages.
    recent = [history[0]] + history[1:][-MAX_TURNS:]

    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=recent,
            temperature=0.3
        )
        reply = response.choices[0].message.content

    except RateLimitError:
        reply = "We are a little busy. Please try again in a minute."

    except AuthenticationError:
        reply = "There is an API authentication problem. Please check the setup."

    except APIConnectionError:
        reply = "I cannot reach the server right now. Please try again shortly."

    history.append({
        "role": "assistant",
        "content": reply
    })

    return reply

In [14]:
print("Total for 6 cupcakes:", get_price("cupcake", 6))
while True:
 user_text = input("You: ")
 if user_text.lower().strip() in ("bye", "exit", "quit"):
  print("Bot: Thank you for visiting Sweet Crumbs!")
 break
 print("Bot:", chat(user_text))

[LOG] get_price('cupcake', 6) -> 480
Total for 6 cupcakes: 480
You: Can I get an eggless chocolate cake?
